1.单进程 语义投影截断代码:

In [ ]:
import os
import csv
import ast
import random
import argparse
import numpy as np
import pandas as pd
from tqdm import tqdm

"""
python PSF.py \
  --dataset_dir /home/wangshuo/resource/datasets/amazon_data/amazon_extend \
  --ablation_csv /home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/allocation_strategy_comparison_ablation_sum.csv \
  --t1_proxy ML3_proxy2_probability \
  --t1_oracle ML3_oracle2_probability \
  --t1_ids product_id_list \
  --t1_low 0.2 \
  --t1_high 0.3 \
  --t2_proxy ML2_proxy4b_probability \
  --t2_oracle ML2_oracle1_probability \
  --t2_ids review_id_list \
  --t2_low 0.2 \
  --t2_high 0.3 \
  --out_csv /home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/PSF_amazon_sum.csv
  """

def safe_literal_eval(val):
    """安全解析 CSV 中存储的字符串列表"""
    if pd.isna(val) or not isinstance(val, str) or val.strip() in ["", "nan", "[]"]:
        return []
    try:
        res = ast.literal_eval(val)
        return res if isinstance(res, list) else []
    except (ValueError, SyntaxError):
        return []

def parse_float_list(lst):
    """将列表转换为浮点数列表"""
    return [float(x) for x in lst if pd.notna(x)]

def load_query_budgets(csv_path, target_frac=0.1, target_method="8_POSSA"):
    """从消融实验 CSV 中读取每个查询在 frac=0.1 下的专属 oracle_cost 预算"""
    query_budgets = {}
    if not os.path.exists(csv_path):
        print(f"[Error] 找不到消融 CSV 文件: {csv_path}")
        return query_budgets

    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                frac = float(row.get('budget_frac', 0))
                if abs(frac - target_frac) > 1e-4: continue
                method = row.get('method', '').strip()
                if method not in ['POSS', '8_POSSA'] and target_method in ['POSS', '8_POSSA']: continue
                if method != target_method and target_method not in ['POSS', '8_POSSA']: continue
                
                q_name = row['query_basename'].strip()
                cost = int(float(row['oracle_cost']))
                if q_name not in query_budgets: query_budgets[q_name] = []
                query_budgets[q_name].append(cost)
            except Exception:
                continue
                
    # 取多轮均值作为该查询的最终公平预算 B
    return {q: int(round(sum(c)/len(c))) for q, c in query_budgets.items()}

def evaluate_instance_double_truncation(
    row, 
    t1_proxy_col, t1_oracle_col, t1_ids_col, t1_low, t1_high,
    t2_proxy_col, t2_oracle_col, t2_ids_col, t2_low, t2_high,
    oracle_cache, budget_used, budget_limit
):
    """
    判断单条核心实例是否通过双截断筛选
    【严格原则】：对包含的多个谓词节点分别、独立检验，绝不相乘！
    """
    # 1. 解析概率列表与节点 ID 列表
    p1_list = parse_float_list(safe_literal_eval(row.get(t1_proxy_col, "[]")))
    o1_list = parse_float_list(safe_literal_eval(row.get(t1_oracle_col, "[]")))
    id1_list = safe_literal_eval(row.get(t1_ids_col, "[]"))

    p2_list = parse_float_list(safe_literal_eval(row.get(t2_proxy_col, "[]")))
    o2_list = parse_float_list(safe_literal_eval(row.get(t2_oracle_col, "[]")))
    id2_list = safe_literal_eval(row.get(t2_ids_col, "[]"))

    # 将所有带谓词的节点组装到一个检查列表中
    nodes_to_check = []
    for idx, (p, o) in enumerate(zip(p1_list, o1_list)):
        nid = id1_list[idx] if idx < len(id1_list) else f"t1_{idx}"
        nodes_to_check.append(("T1", str(nid), p, o, t1_low, t1_high))

    for idx, (p, o) in enumerate(zip(p2_list, o2_list)):
        nid = id2_list[idx] if idx < len(id2_list) else f"t2_{idx}"
        nodes_to_check.append(("T2", str(nid), p, o, t2_low, t2_high))

    calls = 0
    # 2. 逐一、独立检验每一个节点的谓词状态
    for t_name, nid, p_val, o_val, low, high in nodes_to_check:
        
        # A. 硬舍弃 (< low): 只要有一个节点不通过，整个核心实例一票否决！
        if p_val < low:
            return False, budget_used, calls

        # B. 硬接受 (> high): 该节点符合条件，继续检查其他节点
        elif p_val > high:
            continue

        # C. 灰色地带 [low, high]: 需要 Oracle 检验
        else:
            cache_key = (t_name, nid)
            if cache_key in oracle_cache:
                # 缓存命中，不扣减预算
                ok = oracle_cache[cache_key]
            else:
                if budget_used < budget_limit:
                    # 在总预算内，调用 Oracle 真实概率 (>0.5 为有效)
                    ok = o_val > 0.5
                    oracle_cache[cache_key] = ok
                    budget_used += 1
                    calls += 1
                else:
                    # 预算用尽，退化为区间中点 Proxy 硬判定
                    mid = (low + high) / 2.0
                    ok = p_val > mid

            # 如果灰色地带验证失败，同样一票否决！
            if not ok:
                return False, budget_used, calls

    # 走到这里，说明该实例中的 **所有节点** 都分别通过了各自的检验 (AND逻辑成功)
    return True, budget_used, calls

def main():
    parser = argparse.ArgumentParser(description="Double Truncation Baseline on Core Instances")
    parser.add_argument("--dataset_dir", required=True, help="数据集根目录 (e.g., .../amazon_extend)")
    parser.add_argument("--ablation_csv", required=True, help="消融实验 CSV 文件路径")
    
    # Table 1 配置
    parser.add_argument("--t1_proxy", default="ML3_proxy2_probability", help="Table 1 Proxy 列")
    parser.add_argument("--t1_oracle", default="ML3_oracle2_probability", help="Table 1 Oracle 列")
    parser.add_argument("--t1_ids", default="product_id_list", help="Table 1 节点 ID 列名")
    parser.add_argument("--t1_low", type=float, default=0.2, help="Table 1 硬舍弃上限")
    parser.add_argument("--t1_high", type=float, default=0.3, help="Table 1 硬接受下限")

    # Table 2 配置
    parser.add_argument("--t2_proxy", default="ML2_proxy2_probability", help="Table 2 Proxy 列")
    parser.add_argument("--t2_oracle", default="ML2_oracle1_probability", help="Table 2 Oracle 列")
    parser.add_argument("--t2_ids", default="review_id_list", help="Table 2 节点 ID 列名")
    parser.add_argument("--t2_low", type=float, default=0.2, help="Table 2 硬舍弃上限")
    parser.add_argument("--t2_high", type=float, default=0.3, help="Table 2 硬接受下限")

    parser.add_argument("--out_csv", default="Core_Double_Truncation_results.csv", help="最终结果输出 CSV")
    args = parser.parse_args()

    print("[*] 正在加载每个查询的专属 Oracle 预算 (budget_frac = 0.1)...")
    query_budgets = load_query_budgets(args.ablation_csv, target_frac=0.1)
    print(f"[+] 共成功匹配到 {len(query_budgets)} 个查询的预算。")

    agg_dir = os.path.join(args.dataset_dir, "results", "aggregated_results")
    if not os.path.exists(agg_dir):
        print(f"[Error] 找不到目录: {agg_dir}")
        return

    agg_files = sorted([f for f in os.listdir(agg_dir) if f.startswith("aggregated_list_") and f.endswith(".csv")])
    print(f"[*] 在 {agg_dir} 中找到 {len(agg_files)} 个核心实例文件。\n")

    results = []
    
    # 逐个查询处理 (带进度条)
    for fname in tqdm(agg_files, desc="Double Truncation Evaluating", ncols=100):
        q_basename = fname.replace("aggregated_list_", "").replace(".csv", "") + ".graph"
        budget_B = query_budgets.get(q_basename, 500) # 若无配置则使用 500 兜底

        filepath = os.path.join(agg_dir, fname)
        df = pd.read_csv(filepath)
        if df.empty:
            continue

        weight_col = "estimateW" if "estimateW" in df.columns else "a"
        if weight_col not in df.columns:
            continue

        # 将所有的核心实例行打乱，模拟随机访问顺序 (保证公平调用 Oracle 预算)
        indices = list(range(len(df)))
        random.seed(42)
        random.shuffle(indices)

        oracle_cache = {}
        budget_used = 0
        total_oracle_calls = 0
        accepted_weight_sum = 0.0
        accepted_instances_count = 0

        for idx in indices:
            row = df.iloc[idx]
            weight = float(row[weight_col])
            if weight <= 0:
                continue

            # 对单条核心实例内的各个谓词节点分别独立检验
            is_accepted, budget_used, calls = evaluate_instance_double_truncation(
                row=row,
                t1_proxy_col=args.t1_proxy, t1_oracle_col=args.t1_oracle, t1_ids_col=args.t1_ids,
                t1_low=args.t1_low, t1_high=args.t1_high,
                t2_proxy_col=args.t2_proxy, t2_oracle_col=args.t2_oracle, t2_ids_col=args.t2_ids,
                t2_low=args.t2_low, t2_high=args.t2_high,
                oracle_cache=oracle_cache,
                budget_used=budget_used,
                budget_limit=budget_B
            )

            total_oracle_calls += calls
            if is_accepted:
                accepted_weight_sum += weight
                accepted_instances_count += 1

        results.append({
            "query_basename": q_basename,
            "T_hat_double_truncation": round(accepted_weight_sum, 4),
            "total_instances": len(df),
            "accepted_instances": accepted_instances_count,
            "budget_limit_B": budget_B,
            "budget_used": budget_used,
            "total_oracle_calls": total_oracle_calls
        })

    # 保存到输出 CSV
    out_path = os.path.join(args.dataset_dir, "results", "efficiency", args.out_csv)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    
    res_df = pd.DataFrame(results)
    res_df.to_csv(out_path, index=False)
    print(f"\n✅ [双截断基线评估完成]！共处理 {len(res_df)} 个查询。")
    print(f"📁 最终结果已保存至: {out_path}")

if __name__ == "__main__":
    main()

2. 多谓词双阈值

In [ ]:
import os
import csv
import ast
import json
import random
import argparse
import numpy as np
import pandas as pd
from tqdm import tqdm
import concurrent.futures

"""
python PSF.py \
  --parent_dataset amazon_data \
  --dataset amazon_extend \
  --ablation_csv /home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/allocation_strategy_comparison_ablation_sum.csv \
  --table1 product \
  --table1_proxy ML3_proxy2_probability \
  --table1_oracle ML3_oracle2_probability \
  --t1_ids post_id_list \
  --t1_low 0.2 \
  --t1_high 0.6 \
  --table2 review \
  --table2_proxy ML2_proxy2_probability \
  --table2_oracle ML2_oracle1_probability \
  --t2_ids comment_id_list \
  --t2_low 0.2 \
  --t2_high 0.6 \
  --num_workers 16 \
  --out_csv Core_Double_Truncation_amazon_sum.csv

"""


def safe_extract_list(val):
    """【完美适配版】：仅负责把字符串转成 Python List，不强制转换元素类型"""
    if pd.isna(val) or val == "":
        return []
    if isinstance(val, (list, tuple)):
        return list(val)
    if isinstance(val, (int, float)):
        return [val]
    if isinstance(val, str):
        val = val.strip()
        if val in ["", "nan", "[]"]:
            return []
        if val.startswith('[') and val.endswith(']'):
            try:
                res = ast.literal_eval(val)
                return res if isinstance(res, list) else [res]
            except Exception:
                return []
        else:
            return [val]
    return []

def parse_float_list(lst):
    """专门且安全地将概率列表的元素转换为浮点数，忽略无法转换的值"""
    out = []
    for x in lst:
        try:
            out.append(float(x))
        except (ValueError, TypeError):
            pass
    return out

def load_query_budgets(csv_path, target_frac=0.1, target_method="8_POSSA"):
    """从消融实验 CSV 中读取每个查询在 frac=0.1 下的专属 oracle_cost 预算"""
    query_budgets = {}
    if not os.path.exists(csv_path):
        print(f"[Error] 找不到消融 CSV 文件: {csv_path}")
        return query_budgets

    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                frac = float(row.get('budget_frac', 0))
                if abs(frac - target_frac) > 1e-4: continue
                method = row.get('method', '').strip()
                if method not in ['POSS', '8_POSSA'] and target_method in ['POSS', '8_POSSA']: continue
                if method != target_method and target_method not in ['POSS', '8_POSSA']: continue
                
                q_name = row['query_basename'].strip()
                cost = int(float(row['oracle_cost']))
                if q_name not in query_budgets: query_budgets[q_name] = []
                query_budgets[q_name].append(cost)
            except Exception:
                continue
                
    return {q: int(round(sum(c)/len(c))) for q, c in query_budgets.items()}

def evaluate_instance_double_truncation(
    row_dict, 
    t1_proxy_col, t1_oracle_col, t1_ids_col, t1_low, t1_high,
    t2_proxy_col, t2_oracle_col, t2_ids_col, t2_low, t2_high,
    oracle_cache, budget_used, budget_limit
):
    """判断单条核心实例是否通过双截断筛选"""
    p1_list = parse_float_list(safe_extract_list(row_dict.get(t1_proxy_col)))
    o1_list = parse_float_list(safe_extract_list(row_dict.get(t1_oracle_col)))
    id1_list = safe_extract_list(row_dict.get(t1_ids_col))

    p2_list = parse_float_list(safe_extract_list(row_dict.get(t2_proxy_col)))
    o2_list = parse_float_list(safe_literal_eval(row_dict.get(t2_oracle_col)) if 'safe_literal_eval' in globals() else safe_extract_list(row_dict.get(t2_oracle_col)))
    id2_list = safe_extract_list(row_dict.get(t2_ids_col))

    nodes_to_check = []
    for idx, (p, o) in enumerate(zip(p1_list, o1_list)):
        nid = id1_list[idx] if idx < len(id1_list) else f"t1_{idx}"
        nodes_to_check.append(("T1", str(nid), p, o, t1_low, t1_high))

    for idx, (p, o) in enumerate(zip(p2_list, o2_list)):
        nid = id2_list[idx] if idx < len(id2_list) else f"t2_{idx}"
        nodes_to_check.append(("T2", str(nid), p, o, t2_low, t2_high))

    calls = 0
    for t_name, nid, p_val, o_val, low, high in nodes_to_check:
        if p_val < low:
            return False, budget_used, calls
        elif p_val > high:
            continue
        else:
            cache_key = (t_name, nid)
            if cache_key in oracle_cache:
                ok = oracle_cache[cache_key]
            else:
                if budget_used < budget_limit:
                    ok = o_val > 0.5
                    oracle_cache[cache_key] = ok
                    budget_used += 1
                    calls += 1
                else:
                    mid = (low + high) / 2.0
                    ok = p_val > mid

            if not ok:
                return False, budget_used, calls

    return True, budget_used, calls

def process_single_file(fname, agg_dir, query_budgets, args_dict):
    """【子进程核心函数】独立读取文件、计算、返回结果（纯内存操作防死锁）"""
    try:
        q_basename = fname.replace("aggregated_list_", "").replace(".csv", "") + ".graph"
        budget_B = query_budgets.get(q_basename, 500)

        filepath = os.path.join(agg_dir, fname)
        df = pd.read_csv(filepath)
        if df.empty:
            return None

        weight_col = "estimateW" if "estimateW" in df.columns else "a"
        if weight_col not in df.columns:
            return None

        indices = list(range(len(df)))
        random.seed(42)
        random.shuffle(indices)

        oracle_cache = {}
        budget_used = 0
        total_oracle_calls = 0
        accepted_weight_sum = 0.0
        accepted_instances_count = 0

        records = df.to_dict('records')

        for idx in indices:
            row_dict = records[idx]
            weight = float(row_dict[weight_col])
            if weight <= 0:
                continue

            is_accepted, budget_used, calls = evaluate_instance_double_truncation(
                row_dict=row_dict,
                t1_proxy_col=args_dict['t1_proxy'], t1_oracle_col=args_dict['t1_oracle'], t1_ids_col=args_dict['t1_ids'],
                t1_low=args_dict['t1_low'], t1_high=args_dict['t1_high'],
                t2_proxy_col=args_dict['t2_proxy'], t2_oracle_col=args_dict['t2_oracle'], t2_ids_col=args_dict['t2_ids'],
                t2_low=args_dict['t2_low'], t2_high=args_dict['t2_high'],
                oracle_cache=oracle_cache,
                budget_used=budget_used,
                budget_limit=budget_B
            )

            total_oracle_calls += calls
            if is_accepted:
                accepted_weight_sum += weight
                accepted_instances_count += 1

        return {
            "query_basename": q_basename,
            "T_hat_double_truncation": round(accepted_weight_sum, 4),
            "total_instances": len(df),
            "accepted_instances": accepted_instances_count,
            "budget_limit_B": budget_B,
            "budget_used": budget_used,
            "total_oracle_calls": total_oracle_calls
        }
    except Exception as e:
        print(f"\n[Error] 处理文件 {fname} 失败: {e}")
        return None

def main():
    parser = argparse.ArgumentParser(description="Double Truncation Baseline on Core Instances")
    parser.add_argument("--parent_dataset", default="amazon_data")
    parser.add_argument("--dataset", required=True, help="数据集 (e.g., amazon_extend)")
    parser.add_argument("--fastest_bin", default="") 
    parser.add_argument("--ablation_csv", required=True, help="消融实验 CSV 文件路径")
    
    # 【兼容短名字与长名字参数】
    parser.add_argument("--table1", default="product")
    parser.add_argument("--t1_proxy", "--table1_proxy", dest="t1_proxy", default="ML3_proxy2_probability")
    parser.add_argument("--t1_oracle", "--table1_oracle", dest="t1_oracle", default="ML3_oracle2_probability")
    parser.add_argument("--t1_ids", "--table1_ids", dest="t1_ids", default="post_id_list")
    parser.add_argument("--t1_low", type=float, default=0.2)
    parser.add_argument("--t1_high", type=float, default=0.3)

    parser.add_argument("--table2", default="review")
    parser.add_argument("--t2_proxy", "--table2_proxy", dest="t2_proxy", default="ML2_proxy2_probability")
    parser.add_argument("--t2_oracle", "--table2_oracle", dest="t2_oracle", default="ML2_oracle1_probability")
    parser.add_argument("--t2_ids", "--table2_ids", dest="t2_ids", default="comment_id_list")
    parser.add_argument("--t2_low", type=float, default=0.2)
    parser.add_argument("--t2_high", type=float, default=0.3)

    parser.add_argument("--sum_col", default="price")
    parser.add_argument("--sum_label", default="12")
    
    parser.add_argument("--num_workers", type=int, default=8, help="并行处理的核心数")
    parser.add_argument("--out_csv", default="Core_Double_Truncation_results.csv", help="最终结果输出 CSV")
    
    args = parser.parse_args()

    print(f"[*] 正在加载专属 Oracle 预算 (budget_frac = 0.1)...")
    query_budgets = load_query_budgets(args.ablation_csv, target_frac=0.1)
    print(f"[+] 成功匹配 {len(query_budgets)} 个查询预算。")

    base_dir = f"/home/wangshuo/resource/datasets/{args.parent_dataset}/{args.dataset}"
    agg_dir = os.path.join(base_dir, "results", "aggregated_results")
    if not os.path.exists(agg_dir):
        print(f"[Error] 找不到目录: {agg_dir}")
        return

    agg_files = sorted([f for f in os.listdir(agg_dir) if f.startswith("aggregated_list_") and f.endswith(".csv")])
    print(f"[*] 找到 {len(agg_files)} 个核心实例文件。\n")

    if args.out_csv.startswith("/"):
        out_path = args.out_csv
    else:
        out_path = os.path.join(base_dir, "results", "efficiency", args.out_csv)
    
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    fieldnames = [
        "query_basename", "T_hat_double_truncation", "total_instances", 
        "accepted_instances", "budget_limit_B", "budget_used", "total_oracle_calls"
    ]
    with open(out_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    args_dict = vars(args)
    completed_cnt = 0

    print(f"🚀 开始多进程极速评估 (Process Workers = {args.num_workers})...\n")

    with concurrent.futures.ProcessPoolExecutor(max_workers=args.num_workers) as executor:
        futures = {
            executor.submit(process_single_file, fname, agg_dir, query_budgets, args_dict): fname 
            for fname in agg_files
        }

        for future in tqdm(concurrent.futures.as_completed(futures), total=len(agg_files), desc="Progress", ncols=100):
            res = future.result()
            if res is not None:
                with open(out_path, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writerow(res)
                    f.flush() 
                completed_cnt += 1

    print(f"\n✅ [双截断基线评估完成]！共处理并即时保存了 {completed_cnt} 个查询。")
    print(f"📁 最终结果路径: {out_path}")

if __name__ == "__main__":
    main()

3. pro-abae 单进程代码

In [ ]:
import os
import ast
import json
import math
import argparse
import numpy as np
import pandas as pd
from typing import Dict, Tuple
from tqdm import tqdm

# ==========================================
# 1. 核心类：Projection-ABae 采样估计器
# ==========================================
class ProjectionABaeSampler:
    def __init__(self, csv_path: str, T_true_avg: float = None, K: int = 5,
                 post_proxy: str = "ML1_proxy4b_probability",
                 comment_proxy: str = "ML2_oracle2_probability", # 注意：Parler-E 中可能是 ML2_proxy4d2 或类似
                 post_oracle: str = "ML1_oracle2_probability",
                 comment_oracle: str = "ML2_oracle2_probability"):
        
        self.csv_path = csv_path
        self.T_true_avg = T_true_avg
        self.K = K
        
        df = pd.read_csv(csv_path)
        self.instances = self._prepare_instances(df, post_proxy, comment_proxy, post_oracle, comment_oracle)

    def _prepare_instances(self, df: pd.DataFrame, p_proxy_col: str, c_proxy_col: str,
                           p_oracle_col: str, c_oracle_col: str) -> pd.DataFrame:
        if df.empty:
            return pd.DataFrame()

        df = df.copy()
        weight_col = "estimateW" if "estimateW" in df.columns else "a"
        df.rename(columns={weight_col: "a"}, inplace=True)
        df["a"] = pd.to_numeric(df["a"], errors="coerce").fillna(0.0)

        def safe_literal_eval(val):
            if pd.isna(val) or not isinstance(val, str) or val.strip() in ["", "nan", "[]"]:
                return []
            try:
                res = ast.literal_eval(val)
                return res if isinstance(res, list) else []
            except (ValueError, SyntaxError):
                return []

        def to_num_list(lst):
            return [float(x) for x in lst if pd.notna(x)]

        # 解析 ID 与 概率列表
        df["post_ids"] = df["post_id_list"].apply(safe_literal_eval) if "post_id_list" in df.columns else [[] for _ in range(len(df))]
        df["comment_ids"] = df["comment_id_list"].apply(safe_literal_eval) if "comment_id_list" in df.columns else [[] for _ in range(len(df))]

        p_proxy_list = df[p_proxy_col].apply(safe_literal_eval).apply(to_num_list) if p_proxy_col in df.columns else df["a"].apply(lambda x: [])
        c_proxy_list = df[c_proxy_col].apply(safe_literal_eval).apply(to_num_list) if c_proxy_col in df.columns else df["a"].apply(lambda x: [])

        # 1. 计算联合 Proxy 概率 (Proxy = Prod(P_post) * Prod(P_comment))
        df["proxy"] = p_proxy_list.apply(lambda l: float(np.prod(l)) if len(l)>0 else 1.0) * \
                      c_proxy_list.apply(lambda l: float(np.prod(l)) if len(l)>0 else 1.0)

        df["post_oracle_probs"] = df[p_oracle_col].apply(safe_literal_eval).apply(to_num_list) if p_oracle_col in df.columns else df["a"].apply(lambda x: [])
        df["comment_oracle_probs"] = df[c_oracle_col].apply(safe_literal_eval).apply(to_num_list) if c_oracle_col in df.columns else df["a"].apply(lambda x: [])

        # 保留合法实例
        instances = df[df["a"] > 0].reset_index(drop=True)
        return instances

    def _eval_oracle_short_circuit(self, row: pd.Series, oracle_cache: Dict) -> Tuple[int, int]:
        """公平计费的短路 Oracle 验证"""
        post_ids = row.get("post_ids", [])
        comment_ids = row.get("comment_ids", [])
        post_probs = row.get("post_oracle_probs", [])
        comment_probs = row.get("comment_oracle_probs", [])

        calls = 0
        # 1. 验证 Post 节点
        for nid, prob in zip(post_ids, post_probs):
            key = ("post", str(nid))
            if key in oracle_cache:
                ok = oracle_cache[key]
            else:
                ok = float(prob) > 0.5
                oracle_cache[key] = ok
                calls += 1
            if not ok:
                return 0, calls

        # 2. 验证 Comment 节点
        for nid, prob in zip(comment_ids, comment_probs):
            key = ("comment", str(nid))
            if key in oracle_cache:
                ok = oracle_cache[key]
            else:
                ok = float(prob) > 0.5
                oracle_cache[key] = ok
                calls += 1
            if not ok:
                return 0, calls

        return 1, calls

    def stratify_by_proxy_quantile(self, df: pd.DataFrame, K: int) -> pd.DataFrame:
        """ABae 论文核心：按 Proxy 分位数排序并分层"""
        df = df.copy()
        try:
            df["stratum"] = pd.qcut(df["proxy"], K, labels=False, duplicates="drop")
        except Exception:
            df["stratum"] = pd.cut(df["proxy"].rank(method="first"), bins=K, labels=False)
        df["stratum"] = df["stratum"].fillna(0).astype(int)
        return df

    def run_abae_avg(self, total_budget_frac: float = 0.1, pilot_ratio: float = 0.3) -> Dict:
        """
        ABae 论文 Algorithm 1 完整逻辑 (适应于 AVG 估计):
        Stage 1: Pilot 均匀采样，估计 p_k (正例率) 和 sigma_k (标准差)
        Stage 2: 按 sqrt(p_k * sigma_k) 分配剩余预算，再次层内均匀采样
        最终输出: Ratio Estimator (Sum_hat / Count_hat)
        """
        if self.instances.empty:
            return {"T_hat_avg": 0.0, "T_true_avg": self.T_true_avg, "ARE": 0.0, "oracle_cost": 0}

        # 1. ABae 分层 (根据 Proxy 分位数分 K 层)
        df = self.stratify_by_proxy_quantile(self.instances, self.K)
        N_total_pop = len(df)

        # 总物理采样预算 (目标采样核心实例数)
        N_budget = max(1, int(math.floor(total_budget_frac * N_total_pop)))
        N1_pilot = int(math.floor(N_budget * pilot_ratio)) # Pilot 阶段预算 (如 30%)
        N2_stage2 = N_budget - N1_pilot                   # 第二阶段预算 (如 70%)

        oracle_cache = {}
        total_oracle_calls = 0

        # ==========================================
        # === Stage 1: Pilot Sampling (飞行采样) ===
        # ==========================================
        strata_groups = dict(list(df.groupby("stratum")))
        actual_K = len(strata_groups)
        n1_per_stratum = max(1, N1_pilot // actual_K)

        pilot_samples = {}
        pilot_stats = {}

        for k, grp in strata_groups.items():
            Nk = len(grp)
            n1_k = min(n1_per_stratum, Nk)

            # ABae 规则：层内无放回均匀采样
            sample_1 = grp.sample(n1_k, replace=False, random_state=np.random.randint(1 << 30)).copy()
            
            # Oracle 检验与计费
            o_vals, a_vals = [], []
            for _, row in sample_1.iterrows():
                is_valid, calls = self._eval_oracle_short_circuit(row, oracle_cache)
                total_oracle_calls += calls
                o_vals.append(is_valid)
                a_vals.append(row["a"])

            sample_1["oracle"] = o_vals
            sample_1["a_val"] = a_vals
            pilot_samples[k] = sample_1

            # 计算 Pilot 估计量
            p_hat_k = sample_1["oracle"].mean() if len(sample_1) > 0 else 0.0
            valid_a = sample_1[sample_1["oracle"] == 1]["a_val"]
            sigma_hat_k = valid_a.std(ddof=1) if len(valid_a) > 1 else (valid_a.mean() if len(valid_a) == 1 else 0.0)

            pilot_stats[k] = {
                "Nk": Nk,
                "p_hat": p_hat_k,
                "sigma_hat": sigma_hat_k
            }

        # ==========================================
        # === Stage 2: Optimal Allocation & Sampling ===
        # ==========================================
        # ABae 最佳分配权重: T_k ~ sqrt(p_hat_k * sigma_hat_k)
        alloc_weights = {}
        for k, st in pilot_stats.items():
            # 加上 1e-6 容错，防止全 0 导致无法分配预算
            w_k = math.sqrt(max(1e-6, st["p_hat"]) * max(1e-6, st["sigma_hat"])) * st["Nk"]
            alloc_weights[k] = w_k

        sum_w = sum(alloc_weights.values())
        alloc_stage2 = {}
        for k in pilot_stats:
            ratio = (alloc_weights[k] / sum_w) if sum_w > 0 else (1.0 / actual_K)
            alloc_stage2[k] = int(math.floor(N2_stage2 * ratio))

        # 执行 Stage 2 采样并合并估计
        stage2_samples = {}
        for k, grp in strata_groups.items():
            p1_ids = set(pilot_samples[k].index)
            remaining_grp = grp.drop(index=p1_ids, errors="ignore")
            
            n2_k = min(alloc_stage2.get(k, 0), len(remaining_grp))
            if n2_k > 0:
                sample_2 = remaining_grp.sample(n2_k, replace=False, random_state=np.random.randint(1 << 30)).copy()
                o_vals, a_vals = [], []
                for _, row in sample_2.iterrows():
                    is_valid, calls = self._eval_oracle_short_circuit(row, oracle_cache)
                    total_oracle_calls += calls
                    o_vals.append(is_valid)
                    a_vals.append(row["a"])
                sample_2["oracle"] = o_vals
                sample_2["a_val"] = a_vals
                stage2_samples[k] = sample_2
            else:
                stage2_samples[k] = pd.DataFrame(columns=grp.columns)

        # ==========================================
        # === Final Ratio Estimation (分层 HT 估计) ===
        # ==========================================
        total_sum_hat = 0.0
        total_count_hat = 0.0

        for k, grp in strata_groups.items():
            Nk = len(grp)
            # 合并 Stage 1 和 Stage 2 的所有样本
            full_k = pd.concat([pilot_samples[k], stage2_samples[k]], ignore_index=True)
            nk_total = len(full_k)

            if nk_total > 0:
                # 经典的分层 Stratified Uniform Estimator (HT 估计)
                weight_expand = Nk / nk_total
                
                sum_k = (full_k["a_val"] * full_k["oracle"]).sum() * weight_expand
                count_k = full_k["oracle"].sum() * weight_expand

                total_sum_hat += sum_k
                total_count_hat += count_k

        # 计算比率估计值 AVG = SUM / COUNT
        if total_count_hat > 0:
            T_hat_avg = total_sum_hat / total_count_hat
        else:
            T_hat_avg = 0.0

        # 计算 ARE 相对误差
        if self.T_true_avg is not None and self.T_true_avg > 0:
            are = abs(T_hat_avg - self.T_true_avg) / self.T_true_avg
        else:
            are = 0.0

        return {
            "T_hat_avg": float(T_hat_avg),
            "T_true_avg": float(self.T_true_avg) if self.T_true_avg else 0.0,
            "ARE": float(are),
            "oracle_cost": int(total_oracle_calls)
        }

# ==========================================
# 2. 批量评估主主逻辑 (针对 Parler-E 数据集)
# ==========================================
def evaluate_abae_baseline_on_parler_e():
    dataset_name = "dataset_test" # Parler-E 数据集目录名
    base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    agg_dir = os.path.join(base_path, "results", "aggregated_results")
    out_csv = os.path.join(base_path, "results", "efficiency", "Projection_ABae_results_avg.csv")

    # 1. 读取真值 JSON (包含 Sum 和 Count 真实值)
    gt_sum_path = os.path.join(base_path, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability_sum.json")
    gt_cnt_path = os.path.join(base_path, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability_count.json")

    if not os.path.exists(gt_sum_path) or not os.path.exists(gt_cnt_path):
        print(f"[Error] 找不到真值 JSON 文件，请检查路径。")
        return

    with open(gt_sum_path, 'r') as f: gt_sum_map = json.load(f)
    with open(gt_cnt_path, 'r') as f: gt_cnt_map = json.load(f)

    # 2. 计算真实 AVG = SUM / COUNT
    gt_avg_map = {}
    for k, s_val in gt_sum_map.items():
        c_val = gt_cnt_map.get(k, 0)
        q_clean = k.replace(".graph", "")
        if c_val > 0:
            gt_avg_map[q_clean] = s_val / c_val

    # 3. 扫描核心实例 CSV 文件
    agg_files = sorted([f for f in os.listdir(agg_dir) if f.startswith("aggregated_list_") and f.endswith(".csv")])
    print(f"[*] 找到 {len(agg_files)} 个核心实例文件，开始 Projection-ABae 评估...")

    results = []

    for fname in tqdm(agg_files, desc="Evaluating Projection-ABae", ncols=100):
        q_clean = fname.replace("aggregated_list_", "").replace(".csv", "")
        T_true_avg = gt_avg_map.get(q_clean)

        if T_true_avg is None:
            continue

        filepath = os.path.join(agg_dir, fname)

        # 实例化 Projection-ABae 采样器
        sampler = ProjectionABaeSampler(
            csv_path=filepath,
            T_true_avg=T_true_avg,
            K=5,
            post_proxy="ML1_proxy4b_probability",
            comment_proxy="ML2_proxy1_probability", # Parler-E 中的列名
            post_oracle="ML1_oracle2_probability",
            comment_oracle="ML2_oracle2_probability"
        )

        # 运行 ABae (预算设为 0.1，Pilot 比例设为 30%)
        res = sampler.run_abae_avg(total_budget_frac=0.1, pilot_ratio=0.3)

        results.append({
            "query_basename": q_clean + ".graph",
            "T_hat_abae_avg": res["T_hat_avg"],
            "T_true_avg": res["T_true_avg"],
            "ARE": res["ARE"],
            "oracle_cost": res["oracle_cost"]
        })

    # 4. 保存与汇总打印
    res_df = pd.DataFrame(results)
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    res_df.to_csv(out_csv, index=False)

    print("\n" + "=" * 65)
    print(f"📊 Projection-ABae (VLDB 2021) 在 Parler-E (AVG) 上的评估结果汇总")
    print("=" * 65)
    print(f"成功评估查询数           : {len(res_df)}")
    print(f"1. 绝对值相对误差均值 (Mean ARE) : {res_df['ARE'].mean():.4f} ({res_df['ARE'].mean()*100:.2f}%)")
    print(f"2. 绝对值相对误差中位数 (P50 ARE): {res_df['ARE'].median():.4f} ({res_df['ARE'].median()*100:.2f}%)")
    print(f"3. 绝对值相对误差 P90 (P90 ARE)   : {res_df['ARE'].quantile(0.90):.4f} ({res_df['ARE'].quantile(0.90)*100:.2f}%)")
    print(f"4. 绝对值相对误差 P95 (P95 ARE)   : {res_df['ARE'].quantile(0.95):.4f} ({res_df['ARE'].quantile(0.95)*100:.2f}%)")
    print(f"5. 平均 Oracle 开销 (Cost)       : {res_df['oracle_cost'].mean():.1f}")
    print("=" * 65)
    print(f"📁 详细结果已保存至: {out_csv}")

if __name__ == "__main__":
    evaluate_abae_baseline_on_parler_e()

PRO-ABAE 多进程代码

In [ ]:
import os
import ast
import json
import math
import csv
import argparse
import numpy as np
import pandas as pd
from typing import Dict, Tuple
from tqdm import tqdm
import concurrent.futures

#  python PRO-ABAE.py   --parent_dataset amazon_data   --dataset_name amazon_extend   --ablation_csv /home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/allocation_strategy_comparison_ablation_sum.csv   --t1_proxy ML3_proxy2_probability   --t1_oracle ML3_oracle2_probability   --t2_proxy ML2_proxy2_probability   --t2_oracle ML2_oracle1_probability   --workers 16   --out_csv Projection_ABae_amazon_sum.csv

"""
python PRO-ABAE.py \
  --parent_dataset parler_data \
  --dataset_name dataset_test \
  --ablation_csv /home/wangshuo/resource/datasets/parler_data/dataset_test/results/efficiency/allocation_strategy_comparison_ablation_sum.csv \
  --t1_proxy ML1_proxy4b_probability \
  --t1_oracle ML1_oracle2_probability \
  --t2_proxy ML2_proxy1_probability \
  --t2_oracle ML2_oracle2_probability \
  --workers 16 \
  --out_csv Projection_ABae_parler_sum.csv
  
"""

# ==========================================
# 1. 核心类：cc 采样估计器
# ==========================================
class ProjectionABaeSampler:
    def __init__(self, df: pd.DataFrame, T_true: float = None, K: int = 5,
                 post_proxy: str = "ML1_proxy4b_probability",
                 comment_proxy: str = "ML2_proxy1_probability",
                 post_oracle: str = "ML1_oracle2_probability",
                 comment_oracle: str = "ML2_oracle2_probability"):
        
        self.T_true = T_true
        self.K = K
        self.instances = self._prepare_instances(df, post_proxy, comment_proxy, post_oracle, comment_oracle)

    def _prepare_instances(self, df: pd.DataFrame, p_proxy_col: str, c_proxy_col: str,
                           p_oracle_col: str, c_oracle_col: str) -> pd.DataFrame:
        if df.empty: return pd.DataFrame()

        df = df.copy()
        weight_col = "estimateW" if "estimateW" in df.columns else "a"
        df.rename(columns={weight_col: "a"}, inplace=True)
        df["a"] = pd.to_numeric(df["a"], errors="coerce").fillna(0.0)

        def safe_extract_list(val):
            if pd.isna(val) or val == "": return []
            if isinstance(val, str):
                if val.strip() in ["", "nan", "[]"]: return []
                try:
                    res = json.loads(val.replace("'", '"'))
                    return res if isinstance(res, list) else [res]
                except Exception:
                    try:
                        res = ast.literal_eval(val)
                        return res if isinstance(res, list) else [res]
                    except Exception: return []
            return []

        def to_num_list(lst):
            return [float(x) for x in lst if pd.notna(x)]

        # 解析 ID
        id1_col = "post_id_list" if "post_id_list" in df.columns else "product_id_list"
        id2_col = "comment_id_list" if "comment_id_list" in df.columns else "review_id_list"
        df["t1_ids"] = df[id1_col].apply(safe_extract_list) if id1_col in df.columns else [[] for _ in range(len(df))]
        df["t2_ids"] = df[id2_col].apply(safe_extract_list) if id2_col in df.columns else [[] for _ in range(len(df))]

        # 解析概率
        p_proxy_list = df[p_proxy_col].apply(safe_extract_list).apply(to_num_list) if p_proxy_col in df.columns else df["a"].apply(lambda x: [])
        c_proxy_list = df[c_proxy_col].apply(safe_extract_list).apply(to_num_list) if c_proxy_col in df.columns else df["a"].apply(lambda x: [])

        # 联合 Proxy 概率
        df["proxy"] = p_proxy_list.apply(lambda l: float(np.prod(l)) if len(l)>0 else 1.0) * \
                      c_proxy_list.apply(lambda l: float(np.prod(l)) if len(l)>0 else 1.0)

        df["t1_oracle_probs"] = df[p_oracle_col].apply(safe_extract_list).apply(to_num_list) if p_oracle_col in df.columns else df["a"].apply(lambda x: [])
        df["t2_oracle_probs"] = df[c_oracle_col].apply(safe_extract_list).apply(to_num_list) if c_oracle_col in df.columns else df["a"].apply(lambda x: [])

        # 仅保留权重 > 0 的合法实例
        instances = df[df["a"] > 0].reset_index(drop=True)
        return instances

    def _eval_oracle_with_budget(self, row: pd.Series, oracle_cache: Dict, budget_used: int, budget_limit: int) -> Tuple[int, int]:
        """严格受限的 Oracle 验证 (带缓存与短路)"""
        t1_ids, t2_ids = row.get("t1_ids", []), row.get("t2_ids", [])
        t1_probs, t2_probs = row.get("t1_oracle_probs", []), row.get("t2_oracle_probs", [])
        proxy_val = row.get("proxy", 0.0)

        # 构建验证队列
        nodes_to_check = []
        for nid, prob in zip(t1_ids, t1_probs): nodes_to_check.append(("T1", str(nid), prob))
        for nid, prob in zip(t2_ids, t2_probs): nodes_to_check.append(("T2", str(nid), prob))

        calls = 0
        for t_name, nid, o_prob in nodes_to_check:
            key = (t_name, nid)
            if key in oracle_cache:
                ok = oracle_cache[key]
            else:
                if budget_used + calls < budget_limit:
                    ok = float(o_prob) > 0.5
                    oracle_cache[key] = ok
                    calls += 1
                else:
                    # 预算耗尽，退化为 Proxy 硬判决
                    ok = proxy_val > 0.5

            if not ok:
                return 0, calls # 一票否决短路
        return 1, calls

    def stratify_by_proxy_quantile(self, df: pd.DataFrame, K: int) -> pd.DataFrame:
        df = df.copy()
        try:
            df["stratum"] = pd.qcut(df["proxy"], K, labels=False, duplicates="drop")
        except Exception:
            df["stratum"] = pd.cut(df["proxy"].rank(method="first"), bins=K, labels=False)
        df["stratum"] = df["stratum"].fillna(0).astype(int)
        return df

    def run_abae_sum(self, target_budget_frac: float = 0.1, pilot_ratio: float = 0.3, budget_B: int = 500) -> Dict:
        """执行 ABae (SUM 估计)"""
        if self.instances.empty:
            return {"T_hat_sum": 0.0, "ARE": 0.0, "oracle_cost": 0}

        df = self.stratify_by_proxy_quantile(self.instances, self.K)
        N_total_pop = len(df)

        # 按行数计算的阶段划分 (但受物理 budget_B 熔断约束)
        N_rows_budget = max(1, int(math.floor(target_budget_frac * N_total_pop)))
        N1_pilot = int(math.floor(N_rows_budget * pilot_ratio)) 
        N2_stage2 = N_rows_budget - N1_pilot                   

        oracle_cache = {}
        total_budget_used = 0

        # ==========================================
        # === Stage 1: Pilot Sampling ===
        # ==========================================
        strata_groups = dict(list(df.groupby("stratum")))
        actual_K = len(strata_groups)
        n1_per_stratum = max(1, N1_pilot // actual_K)

        pilot_samples = {}
        pilot_stats = {}

        for k, grp in strata_groups.items():
            Nk = len(grp)
            n1_k = min(n1_per_stratum, Nk)

            sample_1 = grp.sample(n1_k, replace=False, random_state=np.random.randint(1 << 30)).copy()
            
            y_vals = []
            for _, row in sample_1.iterrows():
                is_valid, calls = self._eval_oracle_with_budget(row, oracle_cache, total_budget_used, budget_B)
                total_budget_used += calls
                # 核心修正：Y = a * Oracle
                y_vals.append(row["a"] * is_valid)

            sample_1["Y"] = y_vals
            pilot_samples[k] = sample_1

            # 核心修正：计算 Y 的标准差，而非仅正样本的标准差！
            y_series = sample_1["Y"]
            sigma_hat_k = y_series.std(ddof=1) if len(y_series) > 1 else 0.0

            pilot_stats[k] = {
                "Nk": Nk,
                "sigma_hat": sigma_hat_k
            }

        # ==========================================
        # === Stage 2: Neyman Allocation & Sampling ===
        # ==========================================
        alloc_weights = {}
        for k, st in pilot_stats.items():
            # Neyman 分配: W_k = N_k * Sigma_k
            alloc_weights[k] = st["Nk"] * st["sigma_hat"]

        sum_w = sum(alloc_weights.values())
        alloc_stage2 = {}
        for k in pilot_stats:
            ratio = (alloc_weights[k] / sum_w) if sum_w > 0 else (1.0 / actual_K)
            alloc_stage2[k] = int(math.floor(N2_stage2 * ratio))

        stage2_samples = {}
        for k, grp in strata_groups.items():
            p1_ids = set(pilot_samples[k].index)
            remaining_grp = grp.drop(index=p1_ids, errors="ignore")
            
            n2_k = min(alloc_stage2.get(k, 0), len(remaining_grp))
            if n2_k > 0:
                sample_2 = remaining_grp.sample(n2_k, replace=False, random_state=np.random.randint(1 << 30)).copy()
                y_vals = []
                for _, row in sample_2.iterrows():
                    is_valid, calls = self._eval_oracle_with_budget(row, oracle_cache, total_budget_used, budget_B)
                    total_budget_used += calls
                    y_vals.append(row["a"] * is_valid)
                sample_2["Y"] = y_vals
                stage2_samples[k] = sample_2
            else:
                stage2_samples[k] = pd.DataFrame(columns=grp.columns)

        # ==========================================
        # === Final Stratified Estimation (SUM) ===
        # ==========================================
        total_sum_hat = 0.0

        for k, grp in strata_groups.items():
            Nk = len(grp)
            full_k = pd.concat([pilot_samples[k], stage2_samples[k]], ignore_index=True)
            nk_total = len(full_k)

            if nk_total > 0:
                # 经典的分层无偏估计量: Sum( N_k * Mean(Y_k) )
                mean_y_k = full_k["Y"].mean()
                total_sum_hat += mean_y_k * Nk

        are = abs(total_sum_hat - self.T_true) / (self.T_true + 1e-9) if self.T_true else 0.0

        return {
            "T_hat_sum": float(total_sum_hat),
            "ARE": float(are),
            "oracle_cost": int(total_budget_used)
        }

# ==========================================
# 2. 多进程独立计算函数
# ==========================================
def process_single_file(fname, agg_dir, gt_map, query_budgets, args_dict):
    try:
        q_clean = fname.replace("aggregated_list_", "").replace(".csv", "")
        q_basename = q_clean + ".graph"
        T_true = gt_map.get(q_clean)

        if T_true is None: return None

        budget_B = query_budgets.get(q_basename, 500)
        filepath = os.path.join(agg_dir, fname)
        df = pd.read_csv(filepath)

        sampler = ProjectionABaeSampler(
            df=df,
            T_true=T_true,
            K=args_dict["K"],
            post_proxy=args_dict["t1_proxy"],
            comment_proxy=args_dict["t2_proxy"],
            post_oracle=args_dict["t1_oracle"],
            comment_oracle=args_dict["t2_oracle"]
        )

        res = sampler.run_abae_sum(
            target_budget_frac=args_dict["budget_frac"],
            pilot_ratio=args_dict["pilot_ratio"],
            budget_B=budget_B
        )

        return {
            "query_basename": q_basename,
            "T_hat_abae": res["T_hat_sum"],
            "T_true": T_true,
            "ARE": res["ARE"],
            "oracle_cost": res["oracle_cost"],
            "budget_limit_B": budget_B
        }
    except Exception as e:
        print(f"\n[Error] 处理文件 {fname} 失败: {e}")
        return None

# ==========================================
# 3. 批量评估主逻辑 (多进程 + 即时写盘)
# ==========================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset_name", required=True)
    parser.add_argument("--parent_dataset", default="parler_data")
    parser.add_argument("--ablation_csv", required=True)
    parser.add_argument("--t1_proxy", default="ML1_proxy4b_probability")
    parser.add_argument("--t2_proxy", default="ML2_proxy1_probability")
    parser.add_argument("--t1_oracle", default="ML1_oracle2_probability")
    parser.add_argument("--t2_oracle", default="ML2_oracle2_probability")
    
    parser.add_argument("--budget_frac", type=float, default=0.1)
    parser.add_argument("--pilot_ratio", type=float, default=0.3)
    parser.add_argument("--K", type=int, default=5)
    parser.add_argument("--workers", type=int, default=16)
    parser.add_argument("--out_csv", default="Projection_ABae_results_sum.csv")
    args = parser.parse_args()

    base_path = f"/home/wangshuo/resource/datasets/{args.parent_dataset}/{args.dataset_name}"
    agg_dir = os.path.join(base_path, "results", "aggregated_results")
    out_csv_path = os.path.join(base_path, "results", "efficiency", args.out_csv)

    # 1. 加载真值 JSON
    gt_sum_path = os.path.join(base_path, "results", "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")
    if not os.path.exists(gt_sum_path):
        gt_sum_path = os.path.join(base_path, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability_sum.json")
    
    with open(gt_sum_path, 'r') as f: gt_dict = json.load(f)
    gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None and v > 0}

    # 2. 提取公平的 Oracle 预算 (B)
    print("[*] 提取消融实验 CSV 专属预算...")
    query_budgets = {}
    with open(args.ablation_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            frac = float(row.get('budget_frac', 0))
            if abs(frac - args.budget_frac) < 1e-4 and row.get('method') in ['POSS', '8_POSSA']:
                q_name = row['query_basename'].strip()
                cost = int(float(row['oracle_cost']))
                query_budgets.setdefault(q_name, []).append(cost)
    query_budgets = {q: int(round(sum(c)/len(c))) for q, c in query_budgets.items()}

    agg_files = sorted([f for f in os.listdir(agg_dir) if f.startswith("aggregated_list_") and f.endswith(".csv")])

    # 3. 初始化 CSV 表头
    fieldnames = ["query_basename", "T_hat_abae", "T_true", "ARE", "oracle_cost", "budget_limit_B"]
    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    with open(out_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    args_dict = vars(args)
    completed_cnt = 0

    print(f"🚀 开始多进程评估 ABae Baseline (Workers = {args.workers})...\n")

    # 4. 多进程异步提交任务与实时写盘
    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers) as executor:
        futures = {executor.submit(process_single_file, fname, agg_dir, gt_map, query_budgets, args_dict): fname for fname in agg_files}

        for future in tqdm(concurrent.futures.as_completed(futures), total=len(agg_files), desc="Evaluating ABae", ncols=100):
            res = future.result()
            if res is not None:
                with open(out_csv_path, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writerow(res)
                    f.flush() 
                completed_cnt += 1

    print(f"\n✅ [Projection-ABae 评估完成]！已即时保存 {completed_cnt} 个查询。")
    
    # 5. 读取生成好的 CSV 并输出汇总
    res_df = pd.read_csv(out_csv_path)
    if not res_df.empty:
        print("\n" + "=" * 65)
        print(f"📊 Projection-ABae (VLDB 2021) 在 {args.dataset_name} (SUM) 上的最终评估")
        print("=" * 65)
        print(f"1. 绝对值相对误差均值 (Mean ARE) : {res_df['ARE'].mean():.4f} ({res_df['ARE'].mean()*100:.2f}%)")
        print(f"2. 绝对值相对误差中位数 (P50 ARE): {res_df['ARE'].median():.4f} ({res_df['ARE'].median()*100:.2f}%)")
        print(f"3. 绝对值相对误差 P90 (P90 ARE)   : {res_df['ARE'].quantile(0.90):.4f} ({res_df['ARE'].quantile(0.90)*100:.2f}%)")
        print(f"4. 绝对值相对误差 P95 (P95 ARE)   : {res_df['ARE'].quantile(0.95):.4f} ({res_df['ARE'].quantile(0.95)*100:.2f}%)")
        print(f"5. 平均 Oracle 物理开销        : {res_df['oracle_cost'].mean():.1f} (Budget: {res_df['budget_limit_B'].mean():.1f})")
        print("=" * 65)

if __name__ == "__main__":
    main()

In [ ]:
import os
import ast
import json
import math
import csv
import argparse
import numpy as np
import pandas as pd
from typing import Dict, Tuple
from tqdm import tqdm
import concurrent.futures

class ProjectionABaeSampler:
    def __init__(self, df: pd.DataFrame, T_true: float = None, K: int = 5,
                 post_proxy: str = "ML1_proxy4b_probability",
                 comment_proxy: str = "ML2_proxy1_probability",
                 post_oracle: str = "ML1_oracle2_probability",
                 comment_oracle: str = "ML2_oracle2_probability"):
        self.T_true = T_true
        self.K = K
        self.instances = self._prepare_instances(df, post_proxy, comment_proxy, post_oracle, comment_oracle)

    def _prepare_instances(self, df: pd.DataFrame, p_proxy_col: str, c_proxy_col: str,
                           p_oracle_col: str, c_oracle_col: str) -> pd.DataFrame:
        if df.empty: return pd.DataFrame()
        df = df.copy()
        weight_col = "estimateW" if "estimateW" in df.columns else "a"
        df.rename(columns={weight_col: "a"}, inplace=True)
        df["a"] = pd.to_numeric(df["a"], errors="coerce").fillna(0.0)

        def safe_extract_list(val):
            if pd.isna(val) or val == "": return []
            if isinstance(val, str):
                if val.strip() in ["", "nan", "[]"]: return []
                try:
                    res = json.loads(val.replace("'", '"'))
                    return res if isinstance(res, list) else [res]
                except Exception:
                    try:
                        res = ast.literal_eval(val)
                        return res if isinstance(res, list) else [res]
                    except Exception: return []
            return []

        def to_num_list(lst):
            return [float(x) for x in lst if pd.notna(x)]

        id1_col = "post_id_list" if "post_id_list" in df.columns else "product_id_list"
        id2_col = "comment_id_list" if "comment_id_list" in df.columns else "review_id_list"
        df["t1_ids"] = df[id1_col].apply(safe_extract_list) if id1_col in df.columns else [[] for _ in range(len(df))]
        df["t2_ids"] = df[id2_col].apply(safe_extract_list) if id2_col in df.columns else [[] for _ in range(len(df))]

        p_proxy_list = df[p_proxy_col].apply(safe_extract_list).apply(to_num_list) if p_proxy_col in df.columns else df["a"].apply(lambda x: [])
        c_proxy_list = df[c_proxy_col].apply(safe_extract_list).apply(to_num_list) if c_proxy_col in df.columns else df["a"].apply(lambda x: [])

        df["proxy"] = p_proxy_list.apply(lambda l: float(np.prod(l)) if len(l)>0 else 1.0) * \
                      c_proxy_list.apply(lambda l: float(np.prod(l)) if len(l)>0 else 1.0)

        df["t1_oracle_probs"] = df[p_oracle_col].apply(safe_extract_list).apply(to_num_list) if p_oracle_col in df.columns else df["a"].apply(lambda x: [])
        df["t2_oracle_probs"] = df[c_oracle_col].apply(safe_extract_list).apply(to_num_list) if c_oracle_col in df.columns else df["a"].apply(lambda x: [])

        instances = df[df["a"] > 0].reset_index(drop=True)
        return instances

    def _eval_oracle_strict(self, row: pd.Series, oracle_cache: Dict, budget_used: int, budget_limit: int) -> Tuple[int, int, bool]:
        """【修改点】严格的 Oracle 验证，不再使用 Proxy 兜底。预算耗尽返回 out_of_budget=True"""
        t1_ids, t2_ids = row.get("t1_ids", []), row.get("t2_ids", [])
        t1_probs, t2_probs = row.get("t1_oracle_probs", []), row.get("t2_oracle_probs", [])

        nodes_to_check = []
        for nid, prob in zip(t1_ids, t1_probs): nodes_to_check.append(("T1", str(nid), prob))
        for nid, prob in zip(t2_ids, t2_probs): nodes_to_check.append(("T2", str(nid), prob))

        calls = 0
        for t_name, nid, o_prob in nodes_to_check:
            key = (t_name, nid)
            if key in oracle_cache:
                ok = oracle_cache[key]
            else:
                if budget_used + calls < budget_limit:
                    ok = float(o_prob) > 0.5
                    oracle_cache[key] = ok
                    calls += 1
                else:
                    # 🚨 预算耗尽，立即报告熔断，拒绝使用 Proxy 污染无偏性！
                    return 0, calls, True 

            if not ok:
                return 0, calls, False # 不合格，但预算没超
                
        return 1, calls, False

    def stratify_by_proxy_quantile(self, df: pd.DataFrame, K: int) -> pd.DataFrame:
        df = df.copy()
        try:
            df["stratum"] = pd.qcut(df["proxy"], K, labels=False, duplicates="drop")
        except Exception:
            df["stratum"] = pd.cut(df["proxy"].rank(method="first"), bins=K, labels=False)
        df["stratum"] = df["stratum"].fillna(0).astype(int)
        return df

    def run_abae_sum(self, target_budget_frac: float = 0.1, pilot_ratio: float = 0.1, budget_B: int = 500) -> Dict:
        if self.instances.empty:
            return {"T_hat_sum": 0.0, "ARE": 0.0, "oracle_cost": 0}

        df = self.stratify_by_proxy_quantile(self.instances, self.K)
        N_total_pop = len(df)

        # 这里计算出希望采样的行数
        N_rows_budget = max(1, int(math.floor(target_budget_frac * N_total_pop)))
        N1_pilot = int(math.floor(N_rows_budget * pilot_ratio)) 
        N2_stage2 = N_rows_budget - N1_pilot                   

        oracle_cache = {}
        total_budget_used = 0
        budget_exhausted = False

        strata_groups = dict(list(df.groupby("stratum")))
        actual_K = len(strata_groups)
        n1_per_stratum = max(1, N1_pilot // actual_K)

        pilot_samples = {k: [] for k in strata_groups}
        
        # 【修正1】构建每层的迭代器，避免顺序采样带来的饥饿效应
        shuffled_grps = {k: grp.sample(frac=1.0, random_state=np.random.randint(1<<30)) 
                         for k, grp in strata_groups.items()}
        grp_iters = {k: shuffled_grps[k].iterrows() for k in strata_groups}
        
        # === Stage 1: Pilot Sampling (Round-Robin 轮询采样) ===
        for step in range(n1_per_stratum):
            if budget_exhausted: break
            
            # 随机打乱遍历层的顺序，防止固定优先级
            keys = list(strata_groups.keys())
            np.random.shuffle(keys)
            
            for k in keys:
                if budget_exhausted: break
                try:
                    idx, row = next(grp_iters[k])
                except StopIteration:
                    continue
                    
                is_valid, calls, is_out = self._eval_oracle_strict(row, oracle_cache, total_budget_used, budget_B)
                if is_out:
                    budget_exhausted = True
                    break # 这个 row 导致熔断，丢弃它以保证无偏性
                    
                total_budget_used += calls
                pilot_samples[k].append(row["a"] * is_valid)

        # 计算 Pilot 统计量
        pilot_stats = {}
        for k in strata_groups:
            Nk = len(strata_groups[k])
            y_vals = pilot_samples[k]
            sigma_hat = np.std(y_vals, ddof=1) if len(y_vals) > 1 else 0.0
            pilot_stats[k] = {"Nk": Nk, "sigma_hat": sigma_hat}

        # === Stage 2: Neyman Allocation & Sampling (池化打乱采样) ===
        alloc_weights = {k: st["Nk"] * st["sigma_hat"] for k, st in pilot_stats.items()}
        sum_w = sum(alloc_weights.values())
        
        alloc_stage2 = {}
        for k in pilot_stats:
            ratio = (alloc_weights[k] / sum_w) if sum_w > 0 else (1.0 / actual_K)
            alloc_stage2[k] = int(math.floor(N2_stage2 * ratio))

        stage2_samples = {k: [] for k in strata_groups}
        
        if not budget_exhausted:
            # 【修正2】把所有 Stage 2 计划要抽的样本放进一个池子全局打乱
            # 这样如果预算中途耗尽，各层受到的损失是按比例均匀分布的
            stage2_pool = []
            for k in strata_groups:
                n2_k = alloc_stage2.get(k, 0)
                drawn = 0
                while drawn < n2_k:
                    try:
                        idx, row = next(grp_iters[k])
                        stage2_pool.append((k, row))
                        drawn += 1
                    except StopIteration:
                        break
            
            np.random.shuffle(stage2_pool)
            
            for k, row in stage2_pool:
                if budget_exhausted: break
                
                is_valid, calls, is_out = self._eval_oracle_strict(row, oracle_cache, total_budget_used, budget_B)
                if is_out:
                    budget_exhausted = True
                    break
                    
                total_budget_used += calls
                stage2_samples[k].append(row["a"] * is_valid)

        # === Final Stratified Estimation (SUM) ===
        total_sum_hat = 0.0
        sampled_N = 0  # 记录成功被采样覆盖到的层人口数

        for k, grp in strata_groups.items():
            Nk = len(grp)
            y_list = pilot_samples[k] + stage2_samples[k]
            nk_total = len(y_list)

            if nk_total > 0:
                mean_y_k = np.mean(y_list)
                total_sum_hat += mean_y_k * Nk
                sampled_N += Nk
                
        # 【修正3】Stratum Collapsing / Scaling
        # 如果因为预算极其紧张导致某些层完全没被采样(nk_total=0)，不能把它们的总和当成0。
        # 必须利用已采样的代表性总体按比例放大，补全丢失的权重，抵消负偏差。
        if 0 < sampled_N < N_total_pop:
            total_sum_hat = total_sum_hat * (N_total_pop / sampled_N)

        # 防除零保护
        t_true_safe = self.T_true if self.T_true and self.T_true > 0 else 1e-9
        signed_re = (total_sum_hat - t_true_safe) / t_true_safe
        are = abs(total_sum_hat - t_true_safe) / t_true_safe

        return {
            "T_hat_sum": float(total_sum_hat),
            "ARE": float(are),
            "Signed_RE": float(signed_re),
            "oracle_cost": int(total_budget_used)
        }
    
def process_single_file(fname, agg_dir, gt_map, query_budgets, args_dict):
    try:
        q_clean = fname.replace("aggregated_list_", "").replace(".csv", "")
        q_basename = q_clean + ".graph"
        T_true = gt_map.get(q_clean)

        if T_true is None: return None

        budget_B = query_budgets.get(q_basename, 500)
        filepath = os.path.join(agg_dir, fname)
        df = pd.read_csv(filepath)

        sampler = ProjectionABaeSampler(
            df=df,
            T_true=T_true,
            K=args_dict["K"],
            post_proxy=args_dict["t1_proxy"],
            comment_proxy=args_dict["t2_proxy"],
            post_oracle=args_dict["t1_oracle"],
            comment_oracle=args_dict["t2_oracle"]
        )

        res = sampler.run_abae_sum(
            target_budget_frac=args_dict["budget_frac"],
            pilot_ratio=args_dict["pilot_ratio"],
            budget_B=budget_B
        )

        return {
            "query_basename": q_basename,
            "T_hat_abae": res["T_hat_sum"],
            "T_true": T_true,
            "Signed_RE": res["Signed_RE"],
            "ARE": res["ARE"],
            "oracle_cost": res["oracle_cost"],
            "budget_limit_B": budget_B
        }
    except Exception as e:
        print(f"\n[Error] 处理文件 {fname} 失败: {e}")
        return None

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset_name", required=True)
    parser.add_argument("--parent_dataset", default="parler_data")
    parser.add_argument("--ablation_csv", required=True)
    parser.add_argument("--t1_proxy", default="ML1_proxy4b_probability")
    parser.add_argument("--t2_proxy", default="ML2_proxy1_probability")
    parser.add_argument("--t1_oracle", default="ML1_oracle2_probability")
    parser.add_argument("--t2_oracle", default="ML2_oracle2_probability")
    
    parser.add_argument("--budget_frac", type=float, default=0.1)
    parser.add_argument("--pilot_ratio", type=float, default=0.3)
    parser.add_argument("--K", type=int, default=5)
    parser.add_argument("--workers", type=int, default=16)
    parser.add_argument("--out_csv", default="Projection_ABae_results_sum.csv")
    args = parser.parse_args()

    base_path = f"/home/wangshuo/resource/datasets/{args.parent_dataset}/{args.dataset_name}"
    agg_dir = os.path.join(base_path, "results", "aggregated_results")
    out_csv_path = os.path.join(base_path, "results", "efficiency", args.out_csv)

    gt_sum_path = os.path.join(base_path, "results", "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")
    if not os.path.exists(gt_sum_path):
        gt_sum_path = os.path.join(base_path, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability_sum.json")
    
    with open(gt_sum_path, 'r') as f: gt_dict = json.load(f)
    gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None and v > 0}

    query_budgets = {}
    with open(args.ablation_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            frac = float(row.get('budget_frac', 0))
            if abs(frac - args.budget_frac) < 1e-4 and row.get('method') in ['POSS', '8_POSSA']:
                q_name = row['query_basename'].strip()
                cost = int(float(row['oracle_cost']))
                query_budgets.setdefault(q_name, []).append(cost)
    query_budgets = {q: int(round(sum(c)/len(c))) for q, c in query_budgets.items()}

    agg_files = sorted([f for f in os.listdir(agg_dir) if f.startswith("aggregated_list_") and f.endswith(".csv")])

    fieldnames = ["query_basename", "T_hat_abae", "T_true", "Signed_RE", "ARE", "oracle_cost", "budget_limit_B"]
    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    with open(out_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    args_dict = vars(args)
    completed_cnt = 0

    print(f"🚀 开始多进程评估 Strict ABae Baseline (Workers = {args.workers})...\n")

    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers) as executor:
        futures = {executor.submit(process_single_file, fname, agg_dir, gt_map, query_budgets, args_dict): fname for fname in agg_files}

        for future in tqdm(concurrent.futures.as_completed(futures), total=len(agg_files), desc="Evaluating ABae", ncols=100):
            res = future.result()
            if res is not None:
                with open(out_csv_path, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writerow(res)
                    f.flush() 
                completed_cnt += 1
                
    # 汇总计算
    res_df = pd.read_csv(out_csv_path)
    if not res_df.empty:
        mean_sre = res_df['Signed_RE'].mean()
        mean_are = res_df['ARE'].mean()
        print("\n" + "=" * 65)
        print(f"📊 Strict Projection-ABae (Unbiased) 最终评估")
        print("=" * 65)
        print(f"1. 带符号误差均值 (Mean Signed RE): {mean_sre:.4f} ({mean_sre*100:.2f}%)")
        print(f"2. 绝对误差均值   (Mean ARE)      : {mean_are:.4f} ({mean_are*100:.2f}%)")
        print("=" * 65)

if __name__ == "__main__":
    main()

In [ ]:
import os
import ast
import json
import math
import csv
import argparse
import numpy as np
import pandas as pd
from typing import Dict, Tuple
from tqdm import tqdm
import concurrent.futures
import random

class ProjectionABaeSampler:
    def __init__(self, df: pd.DataFrame, T_true: float = None, K: int = 5,
                 post_proxy: str = "ML1_proxy4b_probability",
                 comment_proxy: str = "ML2_proxy1_probability",
                 post_oracle: str = "ML1_oracle2_probability",
                 comment_oracle: str = "ML2_oracle2_probability"):
        self.T_true = T_true
        self.K = K
        self.instances = self._prepare_instances(df, post_proxy, comment_proxy, post_oracle, comment_oracle)

    def _prepare_instances(self, df: pd.DataFrame, p_proxy_col: str, c_proxy_col: str,
                           p_oracle_col: str, c_oracle_col: str) -> pd.DataFrame:
        if df.empty: return pd.DataFrame()
        df = df.copy()
        weight_col = "estimateW" if "estimateW" in df.columns else "a"
        df.rename(columns={weight_col: "a"}, inplace=True)
        df["a"] = pd.to_numeric(df["a"], errors="coerce").fillna(0.0)

        def safe_extract_list(val):
            if pd.isna(val) or val == "": return []
            if isinstance(val, str):
                if val.strip() in ["", "nan", "[]"]: return []
                try:
                    res = json.loads(val.replace("'", '"'))
                    return res if isinstance(res, list) else [res]
                except Exception:
                    try:
                        res = ast.literal_eval(val)
                        return res if isinstance(res, list) else [res]
                    except Exception: return []
            return []

        def to_num_list(lst):
            return [float(x) for x in lst if pd.notna(x)]

        id1_col = "post_id_list" if "post_id_list" in df.columns else "product_id_list"
        id2_col = "comment_id_list" if "comment_id_list" in df.columns else "review_id_list"
        df["t1_ids"] = df[id1_col].apply(safe_extract_list) if id1_col in df.columns else [[] for _ in range(len(df))]
        df["t2_ids"] = df[id2_col].apply(safe_extract_list) if id2_col in df.columns else [[] for _ in range(len(df))]

        p_proxy_list = df[p_proxy_col].apply(safe_extract_list).apply(to_num_list) if p_proxy_col in df.columns else df["a"].apply(lambda x: [])
        c_proxy_list = df[c_proxy_col].apply(safe_extract_list).apply(to_num_list) if c_proxy_col in df.columns else df["a"].apply(lambda x: [])

        df["proxy"] = p_proxy_list.apply(lambda l: float(np.prod(l)) if len(l)>0 else 1.0) * \
                      c_proxy_list.apply(lambda l: float(np.prod(l)) if len(l)>0 else 1.0)

        df["t1_oracle_probs"] = df[p_oracle_col].apply(safe_extract_list).apply(to_num_list) if p_oracle_col in df.columns else df["a"].apply(lambda x: [])
        df["t2_oracle_probs"] = df[c_oracle_col].apply(safe_extract_list).apply(to_num_list) if c_oracle_col in df.columns else df["a"].apply(lambda x: [])

        instances = df[df["a"] > 0].reset_index(drop=True)
        return instances

    def _eval_oracle_strict(self, row: pd.Series, oracle_cache: Dict, budget_used: int, budget_limit: int) -> Tuple[int, int, bool]:
        t1_ids, t2_ids = row.get("t1_ids", []), row.get("t2_ids", [])
        t1_probs, t2_probs = row.get("t1_oracle_probs", []), row.get("t2_oracle_probs", [])

        nodes_to_check = []
        for nid, prob in zip(t1_ids, t1_probs): nodes_to_check.append(("T1", str(nid), prob))
        for nid, prob in zip(t2_ids, t2_probs): nodes_to_check.append(("T2", str(nid), prob))

        calls = 0
        for t_name, nid, o_prob in nodes_to_check:
            key = (t_name, nid)
            if key in oracle_cache:
                ok = oracle_cache[key]
            else:
                if budget_used + calls < budget_limit:
                    ok = float(o_prob) > 0.5
                    oracle_cache[key] = ok
                    calls += 1
                else:
                    return 0, calls, True # 预算耗尽报告熔断

            if not ok:
                return 0, calls, False 
                
        return 1, calls, False

    def stratify_by_proxy_quantile(self, df: pd.DataFrame, K: int) -> pd.DataFrame:
        df = df.copy()
        try:
            df["stratum"] = pd.qcut(df["proxy"], K, labels=False, duplicates="drop")
        except Exception:
            df["stratum"] = pd.cut(df["proxy"].rank(method="first"), bins=K, labels=False)
        df["stratum"] = df["stratum"].fillna(0).astype(int)
        return df

    def run_abae_sum(self, target_budget_frac: float = 0.1, pilot_ratio: float = 0.3, budget_B: int = 500, random_seed: int = 42) -> Dict:
        if self.instances.empty:
            return {"T_hat_sum": 0.0, "ARE": 0.0, "Signed_RE": 0.0, "oracle_cost": 0}

        rng = np.random.default_rng(random_seed)

        df = self.stratify_by_proxy_quantile(self.instances, self.K)
        N_total_pop = len(df)

        N_rows_budget = max(1, int(math.floor(target_budget_frac * N_total_pop)))
        N1_pilot = int(math.floor(N_rows_budget * pilot_ratio)) 
        N2_stage2 = N_rows_budget - N1_pilot                   

        oracle_cache = {}
        total_budget_used = 0
        budget_exhausted = False

        strata_groups = dict(list(df.groupby("stratum")))
        actual_K = len(strata_groups)
        n1_per_stratum = max(1, N1_pilot // actual_K)

        # =================================================================
        # Stage 1: 全局打乱的 Pilot Sampling (防层级饿死)
        # =================================================================
        pilot_candidates = []
        pilot_row_indices = {k: [] for k in strata_groups}
        
        for k, grp in strata_groups.items():
            Nk = len(grp)
            n1_k = min(n1_per_stratum, Nk)
            
            indices = rng.permutation(len(grp))
            sampled_grp = grp.iloc[indices[:n1_k]]
            
            for idx, row in sampled_grp.iterrows():
                pilot_candidates.append((k, idx, row))
                pilot_row_indices[k].append(idx)

        # 全局打乱，保证各层级公平获取预算
        random.seed(random_seed)
        random.shuffle(pilot_candidates)

        pilot_results = {k: [] for k in strata_groups}
        for k, idx, row in pilot_candidates:
            if budget_exhausted: break
            is_valid, calls, is_out = self._eval_oracle_strict(row, oracle_cache, total_budget_used, budget_B)
            if is_out:
                budget_exhausted = True
                break
            total_budget_used += calls
            pilot_results[k].append(row["a"] * is_valid)

        # =================================================================
        # Stage 2: Neyman Allocation & 全局打乱采样
        # =================================================================
        alloc_stage2 = {}
        if not budget_exhausted:
            pilot_stats = {}
            for k, grp in strata_groups.items():
                y_vals = pilot_results[k]
                sigma_hat_k = np.std(y_vals, ddof=1) if len(y_vals) > 1 else 0.0
                pilot_stats[k] = {"Nk": len(grp), "sigma_hat": sigma_hat_k}

            alloc_weights = {k: st["Nk"] * st["sigma_hat"] for k, st in pilot_stats.items()}
            sum_w = sum(alloc_weights.values())
            for k in pilot_stats:
                ratio = (alloc_weights[k] / sum_w) if sum_w > 0 else (1.0 / actual_K)
                alloc_stage2[k] = int(math.floor(N2_stage2 * ratio))

        stage2_candidates = []
        if not budget_exhausted:
            for k, grp in strata_groups.items():
                remaining_grp = grp.drop(index=pilot_row_indices[k], errors="ignore")
                rem_indices = rng.permutation(len(remaining_grp))
                
                # 修复此处：将未定义的 remaining_shuffled 改为 remaining_grp
                n2_k = min(alloc_stage2.get(k, 0), len(remaining_grp))
                sampled_grp = remaining_grp.iloc[rem_indices[:n2_k]]
                
                for idx, row in sampled_grp.iterrows():
                    stage2_candidates.append((k, idx, row))

            random.shuffle(stage2_candidates)

        stage2_results = {k: [] for k in strata_groups}
        for k, idx, row in stage2_candidates:
            if budget_exhausted: break
            is_valid, calls, is_out = self._eval_oracle_strict(row, oracle_cache, total_budget_used, budget_B)
            if is_out:
                budget_exhausted = True
                break
            total_budget_used += calls
            stage2_results[k].append(row["a"] * is_valid)

        # ==========================================
        # === Final Stratified Estimation (SUM) ===
        # ==========================================
        total_sum_hat = 0.0

        for k, grp in strata_groups.items():
            Nk = len(grp)
            all_y_k = pilot_results[k] + stage2_results[k]
            nk_total = len(all_y_k)

            if nk_total > 0:
                mean_y_k = np.mean(all_y_k)
                total_sum_hat += mean_y_k * Nk

        t_true_safe = self.T_true if self.T_true and self.T_true > 0 else 1e-9
        signed_re = (total_sum_hat - t_true_safe) / t_true_safe
        are = abs(total_sum_hat - t_true_safe) / t_true_safe

        return {
            "T_hat_sum": float(total_sum_hat),
            "ARE": float(are),
            "Signed_RE": float(signed_re),
            "oracle_cost": int(total_budget_used)
        }

def process_single_task(fname, run_id, agg_dir, gt_map, query_budgets, args_dict):
    """【子进程核心函数】处理某个 Query 的某次单独运行"""
    try:
        q_clean = fname.replace("aggregated_list_", "").replace(".csv", "")
        q_basename = q_clean + ".graph"
        T_true = gt_map.get(q_clean)

        if T_true is None: return None

        budget_B = query_budgets.get(q_basename, 500)
        filepath = os.path.join(agg_dir, fname)
        df = pd.read_csv(filepath)

        sampler = ProjectionABaeSampler(
            df=df, T_true=T_true, K=args_dict["K"],
            post_proxy=args_dict["t1_proxy"], comment_proxy=args_dict["t2_proxy"],
            post_oracle=args_dict["t1_oracle"], comment_oracle=args_dict["t2_oracle"]
        )

        seed = (abs(hash(q_basename)) % 99999999) + run_id

        res = sampler.run_abae_sum(
            target_budget_frac=args_dict["budget_frac"],
            pilot_ratio=args_dict["pilot_ratio"],
            budget_B=budget_B,
            random_seed=seed
        )

        return {
            "query_basename": q_basename, "run_id": run_id,
            "T_hat_abae": res["T_hat_sum"], "T_true": T_true,
            "Signed_RE": res["Signed_RE"], "ARE": res["ARE"],
            "oracle_cost": res["oracle_cost"], "budget_limit_B": budget_B
        }
    except Exception as e:
        print(f"\n[Error] 处理文件 {fname} (Run {run_id}) 失败: {e}")
        return None

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset_name", required=True)
    parser.add_argument("--parent_dataset", default="parler_data")
    parser.add_argument("--ablation_csv", required=True)
    parser.add_argument("--t1_proxy", default="ML1_proxy4b_probability")
    parser.add_argument("--t2_proxy", default="ML2_proxy1_probability")
    parser.add_argument("--t1_oracle", default="ML1_oracle2_probability")
    parser.add_argument("--t2_oracle", default="ML2_oracle2_probability")
    
    parser.add_argument("--budget_frac", type=float, default=0.1)
    parser.add_argument("--pilot_ratio", type=float, default=0.3)
    parser.add_argument("--K", type=int, default=5)
    parser.add_argument("--runs", type=int, default=1)
    parser.add_argument("--workers", type=int, default=16)
    parser.add_argument("--out_csv", default="Projection_ABae_results_sum.csv")
    args = parser.parse_args()

    base_path = f"/home/wangshuo/resource/datasets/{args.parent_dataset}/{args.dataset_name}"
    agg_dir = os.path.join(base_path, "results", "aggregated_results")
    out_csv_path = os.path.join(base_path, "results", "efficiency", args.out_csv)

    gt_sum_path = os.path.join(base_path, "results", "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")
    if not os.path.exists(gt_sum_path):
        gt_sum_path = os.path.join(base_path, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability_sum.json")
    
    with open(gt_sum_path, 'r') as f: gt_dict = json.load(f)
    gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None and v > 0}

    query_budgets = {}
    with open(args.ablation_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            frac = float(row.get('budget_frac', 0))
            if abs(frac - args.budget_frac) < 1e-4 and row.get('method') in ['POSS', '8_POSSA']:
                q_name = row['query_basename'].strip()
                cost = int(float(row['oracle_cost']))
                query_budgets.setdefault(q_name, []).append(cost)
    query_budgets = {q: int(round(sum(c)/len(c))) for q, c in query_budgets.items()}

    agg_files = sorted([f for f in os.listdir(agg_dir) if f.startswith("aggregated_list_") and f.endswith(".csv")])

    fieldnames = ["query_basename", "run_id", "T_hat_abae", "T_true", "Signed_RE", "ARE", "oracle_cost", "budget_limit_B"]
    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    with open(out_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

    args_dict = vars(args)
    completed_cnt = 0
    total_tasks = len(agg_files) * args.runs

    print(f"🚀 开始多进程独立评估 Strict ABae Baseline (Workers = {args.workers}, Runs per query = {args.runs})...\n")

    with concurrent.futures.ProcessPoolExecutor(max_workers=args.workers) as executor:
        futures = {}
        for fname in agg_files:
            for run_id in range(1, args.runs + 1):
                future = executor.submit(process_single_task, fname, run_id, agg_dir, gt_map, query_budgets, args_dict)
                futures[future] = (fname, run_id)

        for future in tqdm(concurrent.futures.as_completed(futures), total=total_tasks, desc="Evaluating ABae", ncols=100):
            res = future.result()
            if res is not None:
                with open(out_csv_path, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writerow(res)
                    f.flush() 
                completed_cnt += 1
                
    res_df = pd.read_csv(out_csv_path)
    if not res_df.empty:
        mean_sre = res_df['Signed_RE'].mean()
        mean_are = res_df['ARE'].mean()
        print("\n" + "=" * 65)
        print(f"📊 Strict Projection-ABae (Unbiased, 10 runs independent) 最终评估")
        print("=" * 65)
        print(f"有效评估记录数 : {len(res_df)}")
        print(f"1. 带符号误差均值 (Mean Signed RE): {mean_sre:.4f} ({mean_sre*100:.2f}%)")
        print(f"2. 绝对误差均值   (Mean ARE)      : {mean_are:.4f} ({mean_are*100:.2f}%)")
        print("=" * 65)

if __name__ == "__main__":
    main()